# Notebook 36 - Clip-Attribution Correction and Support-Safe Joint Diagnostics

**Framing.** Notebook 35 pursues Hypothesis A: that `np.clip` onto the unit box is the
implementation defect behind the marginal SBC / coverage failures, inherited from
Notebook 34's `primary_action = A_IMPLEMENTATION_FIX_FIRST` and
`clip_sensitivity = material`.

This notebook shows that premise is **provably false**, and that the archived data
already proves it.

> **Lemma (clip-invariance of marginal ranks).** For a truth `t` strictly interior to
> `(0, 1)`: `clip(s, 0, 1) < t` iff `s < t`, and `clip(s, 0, 1) == t` iff `s == t`.
> `figure10_diagnostics._randomized_rank` is built only from `(samples < truth)` and
> `(samples == truth)` counts, so **every marginal rank statistic is bit-identical
> before and after clipping.**

Clipping therefore cannot have caused any marginal failure. Where it *does* bite is
exactly what Notebook 35 marks `NOT_EVALUABLE`:

- **L-C2ST** - `figure10_diagnostics.py:629` clips the posterior samples the classifier
  trains on, with `z_score=False`. Roughly a quarter of those rows carry a literal atom
  at exactly 0.0 or 1.0; the Sobol truth arm carries none.
- **Joint expected coverage** - `_global_case` ranks the log-density of clipped draws
  that sit on the boundary manifold rather than on the flow.

**Everything here is read-only presentation.** Sampling, classifier training, and the
sealed simulation run through
`python -m sleep_sbi.figure10_notebook36_audit --phase all` under `conda activate neurolib`.
No NSF retraining. No writes outside `artifacts/notebook_36/`.

**Scope: synthetic cortical-rate inference only.** No real-EEG posterior is claimed, and
131k/1M scale-up remains NO-GO and is never started here.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path("..").resolve()
N36 = ROOT / "results" / "figure10_8d_7d" / "artifacts" / "notebook_36"
JSON = N36 / "json"
CSV = N36 / "csv"
DIAG = ROOT / "results" / "figure10_8d_7d" / "diagnostics"


def load_json(name):
    path = JSON / name
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else None


def load_csv(name):
    path = CSV / name
    return pd.read_csv(path) if path.is_file() else None


def missing(label):
    display(Markdown(
        f"> **{label} has not been run yet.** Run "
        f"`python -m sleep_sbi.figure10_notebook36_audit --phase all` "
        f"from `S4_sbi/src` under the `neurolib` environment."
    ))


phase0 = load_json("phase0_summary.json")
phase1 = load_json("phase1_summary.json")
phase2a = load_json("phase2a_summary.json")
phase2b = load_json("phase2b_summary.json")
phase3 = load_json("phase3_summary.json")
phase4 = load_json("phase4_summary.json")
lock = load_json("n35_prediction_lock.json")
registry = load_json("clip_sensitivity_registry.json")
protocol = load_json("protocol_lock.json")
print("artifacts present:", [
    name for name, value in [
        ("phase0", phase0), ("phase1", phase1), ("phase2a", phase2a),
        ("phase2b", phase2b), ("phase3", phase3), ("phase4", phase4),
    ] if value is not None
])


artifacts present: ['phase0', 'phase1', 'phase2a', 'phase2b', 'phase3', 'phase4']


## Executive dashboard

In [2]:
lines = ["| # | Claim | Evidence | Status |", "|---|-------|----------|--------|"]

if phase0:
    s = phase0["clip_invariance_summary"]
    lines.append(
        f"| 1 | Clipping is **inert** on every marginal rank statistic | "
        f"worst \\|clip-raw\\| = {s['worst_clip_vs_raw_on_invariant_metrics']:.1e} "
        f"over {s['audit_rows']} archived rows | {phase0['status']} |"
    )
    lines.append(
        f"| 2 | N34's `clip_sensitivity=material` measured **rejection**, not clipping | "
        f"\\|rejection-clip\\| up to "
        f"{s['max_rejection_vs_clip_on_invariant_metrics']:.4f} | PROVEN |"
    )

if phase2a:
    lines.append(
        f"| 3 | Archived L-C2ST `p=0.0` is forced by clipping atoms | "
        f"{phase2a['min_atom_row_rate_posterior']:.1%}-"
        f"{phase2a['max_atom_row_rate_posterior']:.1%} of posterior rows carry an "
        f"atom vs {0.0:.1%} of truths | {phase2a['verdict']} |"
    )

if phase2b:
    rows = pd.DataFrame(phase2b["rows"])
    lines.append(
        f"| 4 | L-C2ST under support-safe sampling | "
        f"p ranges {rows['p_value'].min():.3f}-{rows['p_value'].max():.3f} "
        f"(archived: all 0.0) | {phase2b['verdict']} |"
    )

if phase3:
    ens = [r for r in phase3["comparison"] if r["estimator"] == "ensemble"]
    if ens:
        before = ", ".join(f"{r['joint_ks_clipped']:.4f}" for r in ens)
        after = ", ".join(f"{r['joint_ks_support_safe']:.4f}" for r in ens)
        lines.append(
            f"| 5 | Joint coverage KS (ensemble, 8D/7D) | "
            f"clipped {before} -> support-safe {after} | {phase3['verdict']} |"
        )

if phase4:
    lines.append(
        f"| 6 | Aggregation rule E0 vs E1 vs E2 | "
        f"{'; '.join(phase4['findings'])} | zero-cost arm |"
    )

if protocol:
    power = protocol["sealed_bank"]["power"]
    lines.append(
        f"| 7 | Sealed bank is powered | n={power['n_cases']}, thresholds at "
        f"{power['rank_threshold_in_se']:.1f} SE, H0 false-flag "
        f"{power['false_flag_rate_rank_under_h0']:.4f} | vs 1.2 SE at n=48 |"
    )

display(Markdown("\n".join(lines)))


| # | Claim | Evidence | Status |
|---|-------|----------|--------|
| 1 | Clipping is **inert** on every marginal rank statistic | worst \|clip-raw\| = 0.0e+00 over 270 archived rows | PASS |
| 2 | N34's `clip_sensitivity=material` measured **rejection**, not clipping | \|rejection-clip\| up to 0.1458 | PROVEN |
| 3 | Archived L-C2ST `p=0.0` is forced by clipping atoms | 22.5%-29.4% of posterior rows carry an atom vs 0.0% of truths | ATOM_ARTIFACT_CONFIRMED_SUFFICIENT |
| 4 | L-C2ST under support-safe sampling | p ranges 0.000-0.000 (archived: all 0.0) | LC2ST_FAIL_IS_REAL |
| 5 | Joint coverage KS (ensemble, 8D/7D) | clipped 0.2250, 0.2118 -> support-safe 0.2201, 0.2079 | JOINT_FAIL_IS_REAL_AND_AGGREGATION_LOCALISED |
| 6 | Aggregation rule E0 vs E1 vs E2 | 8d: E0 KS=0.2201, E1 KS=0.2218, E2 KS=0.0301; 7d: E0 KS=0.2079, E1 KS=0.2068, E2 KS=0.0217 | zero-cost arm |
| 7 | Sealed bank is powered | n=512, thresholds at 3.9 SE, H0 false-flag 0.0001 | vs 1.2 SE at n=48 |

## Phase 0 - the clip-invariance lemma

Two checks. A synthetic self-check runs the rank counts before and after clipping in two
leak regimes (one matched to the 0.700-0.786 accept rates observed in the archive, one
far harsher so the check cannot pass vacuously). Then the lemma is re-derived from all
270 rows of the archived `phase2_clip_audit.csv`.


In [3]:
if not phase0:
    missing("Phase 0")
else:
    check = phase0["lemma_self_check"]
    for name, regime in check["regimes"].items():
        print(
            f"self-check [{name:>7}] sigma={regime['proposal_sigma']:.3f} "
            f"leak={regime['leak_fraction']:.1%} "
            f"max|d less|={regime['max_abs_less_count_difference']} "
            f"max|d equal|={regime['max_abs_equal_count_difference']}"
        )
    print()
    table = pd.DataFrame(phase0["clip_invariance_table"])
    display(table[[
        "metric", "lemma_class", "n_rows",
        "clip_vs_raw_max", "rejection_vs_clip_max", "rejection_vs_clip_median",
    ]].round(6))
    display(Markdown(
        "`boundary_point_mass` shows no clip-vs-raw delta only because N34 "
        "computes it for `raw_preclip` on an already-clipped view "
        "(`figure10_notebook34_audit.py:538`); that cell is marked not-applicable "
        "and the informative contrast is clipped "
        f"({phase0['clip_invariance_summary']['boundary_point_mass_clipped_mean']:.4f}) "
        "vs rejection "
        f"({phase0['clip_invariance_summary']['boundary_point_mass_rejection_mean']:.4f})."
    ))


self-check [matched] sigma=0.244 leak=28.1% max|d less|=0 max|d equal|=0
self-check [ stress] sigma=0.450 leak=91.5% max|d less|=0 max|d equal|=0



,metric,lemma_class,n_rows,clip_vs_raw_max,rejection_vs_clip_max,rejection_vs_clip_median
0,rank_mean,CLIP_INVARIANT_PROVEN,90,0.0,0.028062,0.009889
1,central_cov_90,CLIP_INVARIANT_PROVEN,90,0.0,0.145833,0.041667
2,central_cov_95,CLIP_INVARIANT_PROVEN,90,0.0,0.104167,0.041667
3,pit_ecdf_ks,CLIP_INVARIANT_PROVEN,90,0.0,0.077485,0.015290
4,bias_unit,CLIP_INVARIANT_PROVEN,90,0.0,0.013389,0.003352
5,boundary_point_mass,CLIP_SENSITIVE_PROVEN,90,NaN,0.089844,0.040934


`boundary_point_mass` shows no clip-vs-raw delta only because N34 computes it for `raw_preclip` on an already-clipped view (`figure10_notebook34_audit.py:538`); that cell is marked not-applicable and the informative contrast is clipped (0.0426) vs rejection (0.0000).

## Phase 1 - support-safe samplers

Two corrections relative to the Notebook 35 design:

1. **One continuous RNG stream.** N35 replays an archived proposal block and then
   switches to a new seed recipe for the continuation, which breaks the strict pairing
   its matched design rests on.
2. **Mixture labels are redrawn on rejection.** The truncated equal-weight mixture has
   component weights proportional to each member's in-support mass `Z_k`, not `1/K`.
   Holding the label fixed and resampling that member until it accepts preserves `1/K`
   and samples a different distribution - inconsistent with the density used for joint
   log-prob ranks. N35 leaves this undefined.


In [4]:
if not phase1:
    missing("Phase 1")
else:
    toy = phase1["mixture_weight_self_check"]
    bench = phase1["microbenchmark"]
    display(Markdown(
        f"Analytic truncated-mixture weights "
        f"`{np.round(toy['analytic_weights'], 4).tolist()}`; the label-redraw "
        f"sampler realises `{np.round(toy['label_redraw_weights'], 4).tolist()}` "
        f"(max error {toy['label_redraw_max_weight_error']:.2e}). The "
        f"fix-label-then-resample alternative stays pinned at "
        f"`{np.round(toy['fix_label_weights'], 4).tolist()}`, giving a different "
        f"distribution (sd {toy['fix_label_sd']:.5f} vs {toy['truncated_mixture_sd']:.5f})."
    ))
    print(
        f"micro-benchmark: {bench['seconds_per_estimator_case']:.3f} s per "
        f"estimator-case, mean accept {bench['mean_accept_rate']:.4f}, "
        f"boundary mass {bench['max_boundary_point_mass']:.1f}, "
        f"projected Phase 3 {bench['projected_phase3_hours']:.2f} h "
        f"-> {bench['phase3_scope']}"
    )


Analytic truncated-mixture weights `[0.2529, 0.7471]`; the label-redraw sampler realises `[0.2537, 0.7463]` (max error 7.70e-04). The fix-label-then-resample alternative stays pinned at `[0.5046, 0.4954]`, giving a different distribution (sd 0.25513 vs 0.23737).

micro-benchmark: 0.592 s per estimator-case, mean accept 0.6735, boundary mass 0.0, projected Phase 3 2.02 h -> PHASE3_FULL_six_estimators


## Phase 2a - the archived L-C2ST rejections, attributed

The archived classifiers are on disk with their training data. `theta_p` holds the
clipped posterior samples, `theta_q` the Sobol truths. A classifier restricted to a
single boundary-indicator feature separates a quarter of the posterior rows with zero
false positives - so `p = 0.0` is forced by the sampler, independently of whether the
posterior is calibrated.

This is a **sufficiency** result: clipping alone can force rejection. Whether the
posterior is *also* miscalibrated is what Phase 2b decides.


In [5]:
if not phase2a:
    missing("Phase 2a")
else:
    frame = pd.DataFrame(phase2a["rows"])
    ok = frame[frame["status"] == "OK"]
    display(ok[[
        "track", "estimator", "archived_p_value",
        "atom_row_rate_posterior", "atom_row_rate_truth",
        "auc_boundary_indicator_only", "lc2st_style_mse_statistic",
        "permutation_p_value",
    ]].round(5))
    display(Markdown(f"**{phase2a['verdict']}** - {phase2a['interpretation']}"))
    display(Markdown(f"_{phase2a['note']}_"))


,track,estimator,archived_p_value,atom_row_rate_posterior,atom_row_rate_truth,auc_boundary_indicator_only,lc2st_style_mse_statistic,permutation_p_value
0,8d,ensemble,0.0,0.27300,0.0,0.63650,0.03952,0.0005
1,8d,member_1,0.0,0.29450,0.0,0.64725,0.04317,0.0005
2,8d,member_2,0.0,0.27390,0.0,0.63695,0.03967,0.0005
3,8d,member_3,0.0,0.23395,0.0,0.61698,0.03312,0.0005
4,8d,member_4,0.0,0.28790,0.0,0.64395,0.04204,0.0005
5,8d,member_5,0.0,0.26750,0.0,0.63375,0.03860,0.0005
6,7d,ensemble,0.0,0.25155,0.0,0.62578,0.03597,0.0005
7,7d,member_1,0.0,0.24545,0.0,0.62272,0.03497,0.0005
8,7d,member_2,0.0,0.22470,0.0,0.61235,0.03164,0.0005
9,7d,member_3,0.0,0.26155,0.0,0.63078,0.03761,0.0005


**ATOM_ARTIFACT_CONFIRMED_SUFFICIENT** - The archived L-C2ST classifiers were trained on CLIPPED posterior samples with z_score=False. Between 22.5% and 29.4% of posterior rows carry a coordinate at exactly 0.0 or 1.0, while the Sobol truth arm carries none. A classifier restricted to that single indicator separates those rows with zero false positives, so p = 0.0 is forced by the sampling implementation and is not by itself evidence of posterior miscalibration.

_Sufficiency, not necessity: this shows clipping alone can force rejection. Whether the posterior is ALSO miscalibrated is decided by the support-safe rerun in Phase 2b._

## Phase 2b - L-C2ST rerun under support-safe sampling

Everything matches `figure10_diagnostics.run_lc2st` except the sampler: the same
20000-row calibration bank already on disk, the same frozen primary observation, the
same classifier seeds, `num_trials_null=100`, `z_score=False`. Support-safe applies to
both the calibration posterior samples and the `theta_o` local samples, because the
archived path clips both.


In [6]:
if not phase2b:
    missing("Phase 2b")
else:
    frame = pd.DataFrame(phase2b["rows"])
    display(frame[[
        "track", "estimator", "archived_p_value_clipped", "p_value",
        "reject_alpha_0p05", "calibration_cases",
        "extreme_leakage_rows_dropped", "runtime_s",
    ]].round(4))
    display(Markdown(f"**Verdict: {phase2b['verdict']}**"))
    if phase2b["deferred"]:
        display(Markdown(
            f"Deferred for budget: `{', '.join(phase2b['deferred'])}`. "
            f"{phase2b['deferred_reason']}"
        ))


,track,estimator,archived_p_value_clipped,p_value,reject_alpha_0p05,calibration_cases,extreme_leakage_rows_dropped,runtime_s
0,8d,ensemble,0.0,0.0,True,19999,1,3435.5408
1,8d,member_1,0.0,0.0,True,19993,7,3292.1397
2,7d,ensemble,0.0,0.0,True,19999,1,3050.6222
3,7d,member_1,0.0,0.0,True,19998,2,2716.1840


**Verdict: LC2ST_FAIL_IS_REAL**

Deferred for budget: `member_2, member_3, member_4, member_5`. archived cost ~3350 s per estimator; the reduced set keeps the notebook inside its zero-training budget. Use --full-estimators to run all 12 (~11 h, background-runnable).

## Phase 3 - joint coverage under support-safe sampling

Recomputed on `powered_1024` with the archived protocol (1000 draws per case, same rank
definitions), so the KS statistics are directly comparable to the archived 0.2250 (8D)
and 0.2118 (7D).

Truncation rescales the posterior density by a constant depending on `x` only. Joint
log-prob ranks are taken at fixed `x`, so the constant cancels and `norm_posterior=False`
stays valid for injected samples - which is why this is evaluable at all.


In [7]:
if not phase3:
    missing("Phase 3")
else:
    frame = pd.DataFrame(phase3["comparison"])
    frame["ks_delta"] = frame["joint_ks_support_safe"] - frame["joint_ks_clipped"]
    display(frame[[
        "track", "estimator", "cases",
        "joint_ks_clipped", "joint_ks_support_safe", "ks_delta",
        "joint_p_support_safe", "sd_clipped", "sd_support_safe",
    ]].round(5))
    display(Markdown(f"**Verdict: {phase3['verdict']}**"))
    display(Markdown(f"_{phase3['normalizer_note']}_"))
    for track in ("8d", "7d"):
        excluded = load_json(f"phase3_excluded_cases_{track}.json")
        if excluded:
            print(
                f"{track}: {excluded['n_excluded']}/"
                f"{excluded['n_cases_attempted']} cases excluded "
                f"({excluded['reason']}) - {excluded['criterion']}"
            )
    print(f"nflows spline retries: {phase3.get('nflows_spline_retries_this_process')}")


,track,estimator,cases,joint_ks_clipped,joint_ks_support_safe,ks_delta,joint_p_support_safe,sd_clipped,sd_support_safe
0,8d,ensemble,1023,0.22501,0.22005,-0.00496,0.00000,0.63571,0.63554
1,8d,member_1,1023,0.03995,0.04188,0.00193,0.05374,0.63286,0.63601
2,8d,member_2,1023,0.05616,0.05823,0.00207,0.00186,0.63330,0.63342
3,8d,member_3,1023,0.02864,0.03015,0.00150,0.30422,0.58985,0.58753
4,8d,member_4,1023,0.04974,0.04447,-0.00527,0.03395,0.64132,0.64179
5,8d,member_5,1023,0.02928,0.03259,0.00331,0.22257,0.64431,0.64739
6,7d,ensemble,1024,0.21178,0.20789,-0.00388,0.00000,0.56816,0.56244
7,7d,member_1,1024,0.04616,0.04470,-0.00146,0.03241,0.56515,0.56738
8,7d,member_2,1024,0.03014,0.03153,0.00139,0.25520,0.56375,0.56503
9,7d,member_3,1024,0.03397,0.03529,0.00133,0.15225,0.55849,0.55219


**Verdict: JOINT_FAIL_IS_REAL_AND_AGGREGATION_LOCALISED**

_Truncation rescales the posterior density by a constant that depends on x only. Joint log-prob ranks are taken at fixed x, so the constant cancels and norm_posterior=False remains valid for support-safe samples. This is why the archived rank path can consume injected samples directly, and why Notebook 35's NOT_EVALUABLE hedge is unnecessary._

8d: 1/1024 cases excluded (EXTREME_LEAKAGE) - a member could not reach 1000 in-support draws within 128000 proposals, i.e. accept rate below 0.7812%
7d: 0/1024 cases excluded (EXTREME_LEAKAGE) - a member could not reach 1000 in-support draws within 128000 proposals, i.e. accept rate below 0.7812%
nflows spline retries: 0


### Marginal SBC, correctly attributed

Reported for completeness. Per the Phase 0 lemma, clipping is inert here, so any
support-safe change is a **rejection-selection** effect - rejection drops a whole row
whenever *any* coordinate leaks, which reweights the retained marginal even for
coordinates that never leaked. It is not evidence that clipping damaged the marginal.


In [8]:
if not phase3:
    missing("Phase 3")
else:
    marginal = pd.DataFrame(phase3["marginal_comparison"])
    focus = marginal[
        (marginal["estimator"] == "ensemble")
        & (marginal["parameter"].isin(["mui", "mue", "tauA"]))
    ]
    display(focus[[
        "track", "parameter", "rank_mean_clipped", "rank_mean_support_safe",
        "ks_p_clipped", "ks_p_support_safe",
        "holm_clear_issue_clipped", "holm_clear_issue_support_safe",
        "attribution",
    ]].round(5))
    display(Markdown(f"_{phase3['marginal_attribution_note']}_"))


,track,parameter,rank_mean_clipped,rank_mean_support_safe,ks_p_clipped,ks_p_support_safe,holm_clear_issue_clipped,holm_clear_issue_support_safe,attribution
0,8d,mue,0.52069,0.52084,0.00005,0.00003,True,True,REJECTION_SELECTION_NOT_CLIP_REPAIR
1,8d,mui,0.58964,0.58991,0.00000,0.00000,True,True,REJECTION_SELECTION_NOT_CLIP_REPAIR
3,8d,tauA,0.52468,0.52451,0.00004,0.00008,True,True,REJECTION_SELECTION_NOT_CLIP_REPAIR
48,7d,mue,0.51890,0.51832,0.00001,0.00003,True,True,REJECTION_SELECTION_NOT_CLIP_REPAIR
49,7d,mui,0.46076,0.45745,0.00000,0.00000,True,True,REJECTION_SELECTION_NOT_CLIP_REPAIR
51,7d,tauA,0.52042,0.51960,0.00007,0.00002,True,True,REJECTION_SELECTION_NOT_CLIP_REPAIR


_Marginal deltas are reported for completeness only. The Phase 0 lemma proves clipping is inert on every marginal rank statistic, so a support-safe change there is a rejection-selection effect (whole rows dropped when ANY coordinate leaks), not evidence that clipping damaged the marginal._

## Phase 4 - aggregation arms, at zero training cost

Notebook 33 supports an *association* between the equal-weight ensemble and joint
coverage degradation, but not a proven density-dilution root cause. Notebook 35 trains
`member_1` only and concedes it cannot refute ensemble-aggregation failures.

Two aggregation rules on identical checkpoints and identical draws:

- **E0** - truncated equal-weight mixture, `p` proportional to `[(1/K) sum_k q_k] * 1[box]`.
  Its implicit component weights are `Z_k / sum_j Z_j`, so leakier members contribute
  fewer draws.
- **E1** - equal mixture of truncated members, `p = (1/K) sum_k q_k * 1[box] / Z_k`.
  Each member's own truncated posterior gets equal weight. Its sampler is the
  fix-label recipe that is *wrong* for E0 and *right* here.
- **E2** - best single member by joint KS, as a regression reference.


In [9]:
if not phase4:
    missing("Phase 4")
else:
    display(pd.DataFrame(phase4["rows"])[
        ["track", "arm", "cases", "joint_ks", "joint_p", "detail"]
    ].round(5))
    display(Markdown(f"_{phase4['n33_question']}_"))
    display(Markdown(f"Cost: **{phase4['cost']}**."))


,track,arm,cases,joint_ks,joint_p,detail
0,8d,E0_truncated_equal_mixture,1023,0.22005,0.00000,p propto [(1/K) sum_k q_k] * 1[box]
1,8d,E1_equal_mixture_of_truncated,1023,0.22181,0.00000,p = (1/K) sum_k q_k * 1[box] / Z_k
2,8d,E2_best_member_member_3,1023,0.03015,0.30422,best single member by joint KS (regression ref...
3,7d,E0_truncated_equal_mixture,1024,0.20789,0.00000,p propto [(1/K) sum_k q_k] * 1[box]
4,7d,E1_equal_mixture_of_truncated,1024,0.20678,0.00000,p = (1/K) sum_k q_k * 1[box] / Z_k
5,7d,E2_best_member_member_4,1024,0.02170,0.71208,best single member by joint KS (regression ref...


_N33 supports an association between the equal-weight ensemble and joint coverage degradation but not a proven density-dilution root cause. E0 vs E1 separates the two aggregation rules directly._

Cost: **zero new simulations, zero retraining**.

## Phase 5 - powered sealed confirmation

The access-control discipline is taken from the Notebook 35 plan, which gets that part
right. What changes is the size. At n=48 the phenotype thresholds sit near 1.2 standard
errors and reject under H0 roughly a quarter of the time; the bank is one-shot, so an
underpowered read cannot be undone. At n=512 the same thresholds sit beyond 3.9 SE, and
the decision rule is a 10000-resample bootstrap CI rather than a bare threshold.


In [10]:
sealed_root = N36 / "sealed" / "sealed_confirm_512"
prov_path = sealed_root / "sealed_provenance.json"
if not prov_path.is_file():
    missing("Phase 5")
else:
    prov = json.loads(prov_path.read_text(encoding="utf-8"))
    power, small = prov["power"], prov["power_at_n35_size"]
    display(Markdown(
        f"**{prov['bank_id']}** - {prov['cases_per_track']} cases/track, "
        f"generated in {prov.get('generation_wall_s', 0.0) / 60:.1f} min.\n\n"
        f"| n | SE(rank_mean) | threshold in SE | H0 false-flag |\n"
        f"|---|---------------|-----------------|---------------|\n"
        f"| {power['n_cases']} (N36) | {power['se_rank_mean']:.4f} | "
        f"{power['rank_threshold_in_se']:.2f} | "
        f"{power['false_flag_rate_rank_under_h0']:.4f} |\n"
        f"| {small['n_cases']} (N35) | {small['se_rank_mean']:.4f} | "
        f"{small['rank_threshold_in_se']:.2f} | "
        f"{small['false_flag_rate_rank_under_h0']:.4f} |"
    ))
    opened = prov.get("opened_utc")
    consumed = (sealed_root / "sealed_consumed.json").is_file()
    print(f"opened_utc={opened!r}  consumed={consumed}")
    print("seed disjointness:", prov["seed_disjointness"]["disjoint"])


**sealed_confirm_512** - 512 cases/track, generated in 2.6 min.

| n | SE(rank_mean) | threshold in SE | H0 false-flag |
|---|---------------|-----------------|---------------|
| 512 (N36) | 0.0128 | 3.92 | 0.0001 |
| 48 (N35) | 0.0417 | 1.20 | 0.2301 |

opened_utc=None  consumed=False
seed disjointness: True


## Pre-registered rebuttal of Notebook 35

Frozen and hashed in Phase 0, before Notebook 35 completed, so these resolve as genuine
predictions rather than hindsight.


In [11]:
if not lock:
    missing("Phase 0 prediction lock")
else:
    display(Markdown(
        f"Locked at `{lock['created_utc']}`. {lock['purpose']}\n\n"
        f"Notebook 35 state when locked: module present = "
        f"`{lock['n35_state_when_locked']['module_present']}`, artifacts = "
        f"`{lock['n35_state_when_locked']['artifacts_present']}`"
    ))
    for prediction in lock["predictions"]:
        basis = prediction["basis"]
        if isinstance(basis, dict):
            basis = (
                f"at n={basis['n_cases']}, thresholds sit at "
                f"{basis['rank_threshold_in_se']:.2f} SE with an H0 false-flag "
                f"rate of {basis['false_flag_rate_rank_under_h0']:.3f}"
            )
        display(Markdown(
            f"**{prediction['id']}**\n\n"
            f"- Prediction: {prediction['statement']}\n"
            f"- Basis: {basis}\n"
            f"- Falsified if: {prediction['falsified_if']}"
        ))


Locked at `2026-08-06T03:12:14.065674+00:00`. Frozen and hashed before Notebook 35 completes, so Phase 6 can resolve these as genuine predictions rather than hindsight.

Notebook 35 state when locked: module present = `True`, artifacts = `['artifacts\\notebook_35\\json\\environment_report.json', 'artifacts\\notebook_35\\json\\input_artifact_manifest.json', 'artifacts\\notebook_35\\json\\phase0_summary.json', 'artifacts\\notebook_35\\json\\phase1_summary.json', 'artifacts\\notebook_35\\json\\protocol_lock.json', 'artifacts\\notebook_35\\json\\protocol_lock.sha256.json']`

**P1_matched_table_shows_clip_equals_raw**

- Prediction: Notebook 35's matched clipped-vs-rejection table will show clipped and raw_preclip agreeing to float precision on every marginal rank statistic, while rejection differs from both.
- Basis: Clip-invariance lemma; already true across all 270 rows of the N34 archive (worst |clip-raw| = 0.000e+00).
- Falsified if: N35 reports a nonzero clipped-vs-raw delta on rank_mean, pit_ecdf_ks, central_cov_90, or bias_unit.

**P2_residual_registry_misattributes_rejection_as_clip_repair**

- Prediction: N35's Residual Failure Registry will label rejection-induced marginal shifts A_REPAIRED, attributing to clip repair an effect that clipping provably cannot have caused.
- Basis: N35 plan section 6 rule 2 fires when the clipped phenotype fails, the support-safe phenotype is healthy, and the paired delta exceeds CLIP_TOLERANCES. Since clipped == raw exactly, any such delta is a rejection-selection effect.
- Falsified if: N35 explicitly attributes the delta to rejection selection rather than to repairing a clipping defect.

**P3_sealed_48_is_underpowered**

- Prediction: N35's sealed_confirm_48 decision rule is statistically underpowered: its thresholds sit near 1.2 standard errors.
- Basis: at n=48, thresholds sit at 1.20 SE with an H0 false-flag rate of 0.230
- Falsified if: N35 raises the sealed bank size or replaces the bare thresholds with a power-aware rule (e.g. bootstrap CIs).

**P4_lc2st_and_joint_left_unevaluated**

- Prediction: N35 will report L-C2ST and full joint coverage as NOT_EVALUABLE, leaving the only clip-sensitive diagnostics untested.
- Basis: N35 plan section 5 declines both; section 13 lists them as unresolved blockers. Both are in fact evaluable: the 20000-row L-C2ST calibration banks and powered_1024 are already on disk, and truncation rescales the posterior density by an x-only constant, so norm_posterior=False log-prob ranks stay valid.
- Falsified if: N35 reports a support-safe L-C2ST p-value or a support-safe joint coverage KS statistic.

## Scope and claim boundary

**Supported by this notebook**

- Clipping is provably inert on marginal rank statistics; Notebook 34's material clip
  sensitivity measured rejection, not clipping.
- The archived L-C2ST rejections are *sufficiently* explained by clipping atoms.
- Support-safe joint coverage and L-C2ST are evaluable and were evaluated.
- Aggregation rules E0/E1/E2 are separable at zero training and zero simulation cost.
- Extreme-leakage observations exist and are reported, not repaired.

**Not supported, and not claimed**

- Real EEG parameter inversion, or any subject-specific physiological posterior.
- A calibrated mechanistic posterior fit for scientific interpretation.
- That any single member or aggregation rule yields a trusted posterior.
- That 131k or 1M simulations would fix anything; both remain NO-GO here.
